In [8]:
import os
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("m4_01_read_mart_local")
    .master("local[*]")
    .getOrCreate()
)

from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important;}</style>"))

16:19:26 [WARN] o.a.s.s.SparkSession - Using an existing Spark session; only runtime SQL configurations will take effect.


In [9]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("m4_02_sales_daily")
    .getOrCreate()
)

JDBC_URL = (
    f"jdbc:postgresql://{os.getenv('PGHOST','postgres')}:{os.getenv('PGPORT','5432')}/"
    f"{os.getenv('PGDATABASE','dwh')}"
)

wide = (
    spark.read.format("jdbc")
    .option("url", JDBC_URL)
    .option("dbtable", "marts.fct_order_items_wide")
    .option("user", os.getenv("PGUSER","app"))
    .option("password", os.getenv("PGPASSWORD","app"))
    .option("driver", "org.postgresql.Driver")
    .load()
)

16:19:52 [WARN] o.a.s.s.SparkSession - Using an existing Spark session; only runtime SQL configurations will take effect.


In [10]:
INGEST_DATE = "2025-12-03"

sales_daily_df = (
    wide
    .filter(F.col("order_date").isNotNull())
    .withColumn("product_category_name", F.coalesce(F.col("product_category_name"), F.lit("unknown")))
    .withColumn("customer_state", F.coalesce(F.col("customer_state"), F.lit("unknown")))
    .groupBy(
        F.col("order_date").alias("sales_date"),
        F.col("product_category_name"),
        F.col("customer_state"),
    )
    .agg(
        F.countDistinct("order_id").alias("orders_cnt"),
        F.count(F.lit(1)).alias("items_cnt"),
        F.coalesce(F.sum("price"), F.lit(0)).alias("items_revenue"),
        F.coalesce(F.sum("freight_value"), F.lit(0)).alias("freight_revenue"),
        F.coalesce(F.sum(F.col("price") + F.col("freight_value")), F.lit(0)).alias("total_revenue"),
        F.countDistinct("customer_sk").alias("uniq_customers_cnt"),
        F.countDistinct("seller_sk").alias("uniq_sellers_cnt"),
    )
    .withColumn("src_ingest_date", F.lit(INGEST_DATE).cast("date"))
    .withColumn("load_dttm", F.current_timestamp())
)

# в Postgres перед записью:
# begin; truncate table marts.sales_daily; commit;

(
    sales_daily_df
    .write
    .mode("append")
    .format("jdbc")
    .option("url", JDBC_URL)
    .option("dbtable", "marts.sales_daily")
    .option("user", os.getenv("PGUSER","app"))
    .option("password", os.getenv("PGPASSWORD","app"))
    .option("driver", "org.postgresql.Driver")
    .save()
)


In [11]:
wide.show(10,0)

+-------------+--------------------------------+-------------+-----------+----------+---------+-------------------+----------+-------------------+----------------------+---------------------+---------------------+---------------------+------------+-------------+--------------+---------------+-------------------------+
|order_item_sk|order_id                        |order_item_id|customer_sk|product_sk|seller_sk|order_approved_at  |order_date|shipping_limit_date|price                 |freight_value        |product_category_name|seller_city          |seller_state|customer_city|customer_state|src_ingest_date|load_dttm                |
+-------------+--------------------------------+-------------+-----------+----------+---------+-------------------+----------+-------------------+----------------------+---------------------+---------------------+---------------------+------------+-------------+--------------+---------------+-------------------------+
|11           |263ba12390d0fbce329dd16da